In [1]:
#URl : https://www.kaggle.com/competitions/playground-series-s6e4/overview

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder,StandardScaler
from sklearn.model_selection import cross_validate,train_test_split,StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score,accuracy_score,roc_auc_score

from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings

In [3]:
warnings.simplefilter(action = 'ignore')

In [ ]:
full_data = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/Predicting_Irrigation_Need/train.csv',index_col = 'id')

In [5]:
print((full_data['Irrigation_Need'] == 'Low').sum()/629999)
print((full_data['Irrigation_Need'] == 'Medium').sum()/629999)
print((full_data['Irrigation_Need'] == 'High').sum()/629999)

0.5871707732869417
0.3794831420367334
0.03334767198043172


We can see that label of dataset is been imbalanced, so we need to use roc_auc_score and some fixing imbalanced approach

In [6]:
#X_train,X_valid,y_train,y_valid = train_test_split(full_data.drop(columns = 'Irrigation_Need'),full_data['Irrigation_Need'],
#                                                   test_size = 0.2,stratify = full_data['Irrigation_Need'],random_state = 42)

In [7]:
X_train = full_data.drop(columns = 'Irrigation_Need')
y_train = full_data['Irrigation_Need'].map({
    'Low' : 0,
    'Medium' : 1,
    'High' : 2
})

In [8]:
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        res = pd.unique(X_train[col])
        if res.shape[0] <= 10:
            print(f"{col} {res}")

Soil_Type ['Loamy' 'Clay' 'Sandy' 'Silt']
Crop_Type ['Sugarcane' 'Wheat' 'Rice' 'Potato' 'Cotton' 'Maize']
Crop_Growth_Stage ['Sowing' 'Vegetative' 'Flowering' 'Harvest']
Season ['Zaid' 'Kharif' 'Rabi']
Irrigation_Type ['Drip' 'Rainfed' 'Sprinkler' 'Canal']
Water_Source ['Rainwater' 'River' 'Reservoir' 'Groundwater']
Mulching_Used ['No' 'Yes']
Region ['East' 'South' 'North' 'West' 'Central']


In [9]:
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        res = pd.unique(X_train[col])
        print(f"{col} {res.shape[0]}")

Soil_Type 4
Crop_Type 6
Crop_Growth_Stage 4
Season 3
Irrigation_Type 4
Water_Source 4
Mulching_Used 2
Region 5


In [10]:
num_cols = [col for col in X_train.columns if X_train[col].dtype in ['int','float']]
cat_cols = [col for col in X_train.columns if col not in num_cols]
cat_OH_cols = ['Region','Water_Source','Irrigation_Type','Season']
cat_OR_cols = [col for col in cat_cols if col not in cat_OH_cols]

In [11]:
preprocess = ColumnTransformer(transformers = [
    ('num',StandardScaler(copy = True,with_mean = True,with_std = True),num_cols),
    ('cat_OH',OneHotEncoder(drop='first',sparse_output = True,handle_unknown = 'ignore'),cat_OH_cols),
    ('cat_OR',OrdinalEncoder(handle_unknown = 'use_encoded_value',unknown_value = -1),cat_OR_cols)
])

sample_weight = compute_sample_weight(class_weight = 'balanced',y = y_train)

In [12]:
xgb_model = Pipeline(steps = [
    ('preprocess',preprocess),
    ('model',XGBClassifier(
        n_estimators = 1500,
        objective = 'multi:softprob',
        num_class = 3,
        max_depth = 8,
        learning_rate = 0.01,
        subsample = 0.8,
        colsample_bytree = 0.8,
        random_state = 42,
        n_jobs = -1
    ))
])
catboost_model = CatBoostClassifier(
    iterations = 1500,
    learning_rate = 0.01,
    depth = 8,
    l2_leaf_reg = 3,
    loss_function = 'multiclass',
    eval_metric = 'multiclass',
    random_state = 42
)
light_model = Pipeline(steps = [
    ('preprocess',preprocess),
    ('model',LGBMClassifier(
        objective='multiclass',  # 'binary' cho binary classification
        num_class=3,             # chỉ cần với multi-class
        boosting_type='gbdt',    # hoặc 'dart', 'goss'
        learning_rate=0.01,
        n_estimators = 1500,
        max_depth= 8,             # -1 = không giới hạn
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=0.0,
        random_state=42
    ))
])
model_lists = {
    "XGBoost": xgb_model,
    "LightGBM": light_model,
}

In [13]:
X_sample,_,y_sample,_ = train_test_split(X_train,y_train,train_size = 0.05,stratify = y_train,random_state = 42)
kf = StratifiedKFold(n_splits = 5,shuffle = True,random_state = 42)
result = {}
for name,model in model_lists.items():
    scores = cross_validate(
        model,
        X_sample,y_sample,
        cv = kf,
        scoring = ['accuracy', 'roc_auc_ovr']
    )
    result[name] = scores

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006192 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2713
[LightGBM] [Info] Number of data points in the train set: 25200, number of used features: 27
[LightGBM] [Info] Start training from score -0.532487
[LightGBM] [Info] Start training from score -0.968838
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001565 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2711
[LightGBM] [Info] Number of data points in the train set: 25200, number of used features: 27
[LightGBM] [Info] Start training from score -0.532420
[LightGBM] [Info] Start training from score -0.968943
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Auto-choosing co

In [17]:
for name, score_dict in result.items():
    print(f"=== {name} ===")
    for metric, values in score_dict.items():
        print(f"{metric}: {values.mean():.4f} ± {values.std():.4f}")
    print()

=== XGBoost ===
fit_time: 211.0736 ± 76.6156
score_time: 3.9182 ± 1.1917
test_accuracy: 0.9833 ± 0.0011
test_roc_auc_ovr: 0.9963 ± 0.0009

=== LightGBM ===
fit_time: 23.3714 ± 1.2123
score_time: 8.5418 ± 0.4678
test_accuracy: 0.9831 ± 0.0005
test_roc_auc_ovr: 0.9963 ± 0.0008



In [18]:
light_model.fit(X_train,y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.035775 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2713
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 27
[LightGBM] [Info] Start training from score -0.532441
[LightGBM] [Info] Start training from score -0.968947
[LightGBM] [Info] Start training from score -3.400769


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Soil_pH', 'Soil_Moisture',
                                                   'Organic_Carbon',
                                                   'Electrical_Conductivity',
                                                   'Temperature_C', 'Humidity',
                                                   'Rainfall_mm',
                                                   'Sunlight_Hours',
                                                   'Wind_Speed_kmh',
                                                   'Field_Area_hectare',
                                                   'Previous_Irrigation_mm']),
                                                 ('cat_OH',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['Region', 'Water_Source',
                                                   'Irrigation_Type',
                                                   'Season']),
                                                 ('cat_OR',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Soil_Type', 'Crop_Type',
                                                   'Crop_Growth_Stage',
                                                   'Mulching_Used'])])),
                ('model',
                 LGBMClassifier(colsample_bytree=0.8, learning_rate=0.01,
                                max_depth=8, n_estimators=1500, num_class=3,
                                objective='multiclass', random_state=42,
                                subsample=0.8))])

In [21]:
X_test = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/Predicting_Irrigation_Need/test.csv',index_col = 'id')
preds = light_model.predict_proba(X_test)

In [22]:
preds = pd.Series(preds.argmax(axis = 1)).map({
    0 : 'Low',
    1 : 'Medium',
    2 : 'High'
})
submission = pd.DataFrame({
    'id' : X_test.index,
    'Irrigation_Need' : preds
})
submission.to_csv('submission.csv',index = False)